# Convert prediction JSON to a DataFrame and LaTeX longtable

This notebook aligns one result document with its source sentences, displays the aligned DataFrame directly in Jupyter, and converts that same DataFrame into the publication-ready LaTeX longtable.

In [31]:
import json
import re
from difflib import SequenceMatcher
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display
from spacy.symbols import VERB
from spacy.tokens import Doc
from evaluation_helpers import nlp
from utils.helpers import find_project_root

PROJECT_ROOT = find_project_root()
DOC_ID = 105
SELECTED_MODEL = "gpt-5.4"
SELECTED_DATASET = "cooking"

## Align Strategy: Align results to original sentence

### NL2P predictions
- Predictions are processed in their original order.
- Only the first word of each predicted verb phrase is matched. (avoid phrasal verbs)
- Matching is case-insensitive but requires the exact word form and complete word boundaries.
- It searches from the previously matched sentence onward and never moves backward.
- Multiple predictions can match the same sentence.
- For NL2P-Coref, matching uses the coreference-resolved sentence.

### Dataset Actions
- Each action uses its global act_idx word position.
- All original sentences are flattened into one word sequence.
- The source word at act_idx must exactly equal the annotated verb.
- Cumulative sentence boundaries determine which sentence contains that index.
- Nested argument lists are flattened after alignment.
- No phrase matching, scoring, or lemmatization is used.

### GPT3-to-Plan
- Actions are processed in their original order.
- Only the first word from the action’s verb field is used. (avoid phrasal verbs)
- The verb and candidate sentence words are lemmatized with spaCy using explicit POS=VERB and verb morphology.
> GPT3-to-Plan does not consistently lemmatize every verb, but it frequently outputs normalized or canonical action names rather than exact source spans. Lemmatization is used only internally to determine sentence alignment and would not affect the output in table
- It selects the first matching sentence from the current position onward.
- The cursor never moves backward, but multiple actions can occupy one sentence.
- Arguments are not used for matching.
- No sentence scoring is performed.
- GPT3-to-Plan is matched against the original sentences, not coreference-resolved sentences.


## Note:
Due to the first-word match strategy, NL2P predictions and GPT3-to-Plan results may aligned to wrong sentence which has other verbs shared the same first word. This may require manual fix in the final output.

Some Fallbacks:
- If a verb cannot be found at or after the current position, it falls back to searching from the beginning of the text, preventing it from incorrectly skipping over valid predictions.
- If a verb cannot be aligned to any sentence, it maps that prediction to the current sentence it is processing.

In [32]:
LATEX_ESCAPES = {
    "\\": r"\textbackslash{}",
    "&": r"\&",
    "%": r"\%",
    "$": r"\$",
    "#": r"\#",
    "_": r"\_",
    "{": r"\{",
    "}": r"\}",
    "~": r"\textasciitilde{}",
    "^": r"\textasciicircum{}",
}


def escape_latex(value):
    """Escape LaTeX special characters in table content."""
    return "".join(LATEX_ESCAPES.get(char, char) for char in str(value))


def phrase_occurs(phrase, sentence):
    "Return whether the exact first word of a verb phrase occurs in a sentence."
    words = str(phrase).split()
    if not words:
        return False

    first_word = re.escape(words[0])
    pattern = r"(?<!\w)" + first_word + r"(?!\w)"
    return re.search(pattern, sentence, flags=re.IGNORECASE) is not None


def align_predictions_to_sentences(sentences, predictions):
    """Align ordered predictions to the first matching sentence at or after the current one."""
    aligned = [[] for _ in sentences]
    sentence_index = 0

    for prediction in predictions:
        verb = prediction.get("verb", "")
        match_index = next(
            (
                index
                for index in range(sentence_index, len(sentences))
                if phrase_occurs(verb, sentences[index])
            ),
            None,
        )
        if match_index is None:
            match_index = next(
                (
                    index
                    for index in range(0, sentence_index)
                    if phrase_occurs(verb, sentences[index])
                ),
                None,
            )
        if match_index is None:
            # Fallback for completely hallucinated/unmatched verbs: attach to current sentence
            match_index = sentence_index

        aligned[match_index].append(prediction)
        sentence_index = match_index

    return aligned


def flatten_action_arguments(arguments):
    """Flatten either nested gold arguments or flat predicted arguments."""
    flattened = []
    for argument in arguments or []:
        if isinstance(argument, list):
            flattened.extend(argument)
        else:
            flattened.append(argument)
    return [str(argument) for argument in flattened]


def align_gold_actions_to_sentences(sentences, gold_actions):
    """Align gold actions by their exact global source-word index."""
    aligned = [[] for _ in sentences]
    words = []
    sentence_ends = []
    for sentence in sentences:
        words.extend(sentence.split())
        sentence_ends.append(len(words))

    for action in gold_actions:
        act_idx = action.get("act_idx")
        if not isinstance(act_idx, int) or not 0 <= act_idx < len(words):
            raise ValueError(f"Invalid gold act_idx: {act_idx!r}")
        source_verb = words[act_idx]
        if source_verb != str(action.get("verb")):
            raise ValueError(
                f"Gold verb {action.get('verb')!r} does not match source token "
                f"{source_verb!r} at index {act_idx}."
            )
        sentence_index = next(
            index for index, end in enumerate(sentence_ends) if act_idx < end
        )
        aligned[sentence_index].append(
            {
                "verb": source_verb,
                "arguments": flatten_action_arguments(
                    action.get("arguments", [])
                ),
            }
        )
    return aligned


@lru_cache(maxsize=None)
def lemmatize_as_verb(word):
    "Lemmatize one word with spaCy while explicitly assigning verb POS."
    word = str(word)
    parsed = nlp(word)
    parsed_verb = next(
        (token for token in parsed if token.pos == VERB),
        None,
    )
    if parsed_verb is not None:
        return parsed_verb.lemma_.casefold()

    lower = word.casefold()
    document = Doc(nlp.vocab, words=[word])
    token = document[0]
    token.pos = VERB

    if lower.endswith("ing"):
        token.tag_ = "VBG"
        token.set_morph("Aspect=Prog|Tense=Pres|VerbForm=Part")
    elif lower.endswith("ed"):
        token.tag_ = "VBD"
        token.set_morph("Tense=Past|VerbForm=Fin")
    elif lower.endswith("s") and not lower.endswith("ss"):
        token.tag_ = "VBZ"
        token.set_morph(
            "Number=Sing|Person=3|Tense=Pres|VerbForm=Fin"
        )
    else:
        token.tag_ = "VB"
        token.set_morph("VerbForm=Inf")

    nlp.get_pipe("lemmatizer")(document)
    return token.lemma_.casefold()


def comparison_verb_occurs(verb, sentence):
    "Match only the first GPT3-to-Plan verb word by its verb lemma."
    words = str(verb).split()
    if not words:
        return False

    verb_lemma = lemmatize_as_verb(words[0])
    sentence_words = re.findall(
        r"[A-Za-z]+(?:[-\u2019\x27][A-Za-z]+)*",
        str(sentence),
    )
    return any(
        lemmatize_as_verb(word) == verb_lemma
        for word in sentence_words
    )


def align_comparison_actions_to_sentences(sentences, actions):
    "Align GPT3-to-Plan actions to the first sentence with the verb lemma."
    aligned = [[] for _ in sentences]
    sentence_index = 0

    for action in actions:
        verb = action.get("verb", "")
        match_index = next(
            (
                index
                for index in range(sentence_index, len(sentences))
                if comparison_verb_occurs(verb, sentences[index])
            ),
            None,
        )
        if match_index is None:
            match_index = next(
                (
                    index
                    for index in range(0, sentence_index)
                    if comparison_verb_occurs(verb, sentences[index])
                ),
                None,
            )
        if match_index is None:
            # Fallback for completely hallucinated/unmatched verbs: attach to current sentence
            match_index = sentence_index

        aligned[match_index].append(action)
        sentence_index = match_index

    return aligned


def read_comparison_actions(comparison_path, dataset, doc_id, sentences):
    """Load and align the matching GPT3-to-Plan result document."""
    with Path(comparison_path).open(encoding="utf-8") as file:
        documents = json.load(file)
    comparison = next(
        (
            item
            for item in documents
            if item.get("dataset") == dataset
            and str(item.get("doc_id")) == str(doc_id)
        ),
        None,
    )
    if comparison is None:
        raise ValueError(
            f"GPT3-to-Plan record for {dataset!r} document {doc_id!r} was not found."
        )
    if comparison.get("sentences") != sentences:
        raise ValueError("GPT3-to-Plan and NL2P sentence lists do not match.")
    return align_comparison_actions_to_sentences(
        sentences, comparison.get("prediction", [])
    )


def format_action_list(actions):
    """Format actions compactly inside one comparison column."""
    if not actions:
        return r"\textit{--}"
    formatted = []
    for action in actions:
        verb_text = escape_latex(action.get("verb", ""))
        verb = rf"\textbf{{{verb_text}}}"
        arguments = flatten_action_arguments(action.get("arguments", []))
        argument_text = "(" + ", ".join(
            escape_latex(argument) for argument in arguments
        ) + ")"
        formatted.append(f"[{verb}, {argument_text}]")
    return r"\par ".join(formatted)


def project_text_position(position, opcodes):
    """Project a character position from original text into resolved text."""
    for tag, original_start, original_end, resolved_start, resolved_end in opcodes:
        if position < original_start:
            return resolved_start
        if original_start <= position <= original_end:
            if tag == "equal":
                return resolved_start + position - original_start
            if original_end == original_start:
                return resolved_start
            fraction = (position - original_start) / (original_end - original_start)
            return round(resolved_start + fraction * (resolved_end - resolved_start))
    return opcodes[-1][4]


def align_coref_sentences(original_sentences, original_text, resolved_text):
    """Project original sentence boundaries into the coreference-resolved text."""
    original_starts = []
    cursor = 0
    for sentence in original_sentences:
        start = original_text.find(sentence, cursor)
        if start < 0:
            raise ValueError(f"Could not locate original sentence: {sentence!r}")
        original_starts.append(start)
        cursor = start + len(sentence)

    opcodes = SequenceMatcher(
        None, original_text, resolved_text, autojunk=False
    ).get_opcodes()
    resolved_starts = [
        project_text_position(start, opcodes) for start in original_starts
    ]
    resolved_starts.append(len(resolved_text))

    return [
        resolved_text[start:end].strip() or None
        for start, end in zip(resolved_starts, resolved_starts[1:])
    ]


def read_coref_sentences(coref_path, dataset, doc_id, original_sentences):
    """Read one JSONL coreference record and align it to original sentences."""
    with Path(coref_path).open(encoding="utf-8") as file:
        record = next(
            (
                item
                for line in file
                if (item := json.loads(line)).get("domain") == dataset
                and str(item.get("doc_id")) == str(doc_id)
            ),
            None,
        )
    if record is None:
        raise ValueError(
            f"Coreference record for {dataset!r} document {doc_id!r} was not found."
        )

    return align_coref_sentences(
        original_sentences,
        record["original_text"],
        record["resolved_text"],
    )


In [33]:
FIXED_RESULT_COLUMNS = {
    "Sentence",
    "Corefed Sentence",
    "Dataset Actions",
    "GPT3-to-Plan",
}


def get_results_column(dataframe):
    "Return the single model-result column in an aligned result DataFrame."
    result_columns = [
        column for column in dataframe.columns
        if column not in FIXED_RESULT_COLUMNS
    ]
    if len(result_columns) != 1:
        raise ValueError(
            "Expected exactly one model-result column, found "
            f"{result_columns!r}."
        )
    return result_columns[0]


def display_wrapped_dataframe(dataframe):
    """Display a fixed-width DataFrame with fully wrapped cell content."""
    with pd.option_context("display.max_colwidth", None):
        display(
            dataframe.style
            .set_properties(
                **{
                    "white-space": "normal",
                    "overflow-wrap": "anywhere",
                    "vertical-align": "top",
                    "text-align": "left",
                }
            )
            .set_table_styles(
                [{
                    "selector": "table",
                    "props": [
                        ("table-layout", "fixed"),
                        ("width", "100%"),
                    ],
                }]
            )
        )


def json_result_to_dataframe(
    input_path,
    results_name,
    doc_id,
    comparison_path,
    coref_path=None,
):
    "Read and align one result document into a structured DataFrame."
    input_path = Path(input_path)
    with input_path.open(encoding="utf-8") as file:
        documents = json.load(file)

    if isinstance(documents, dict):
        documents = [documents]

    document = next(
        (item for item in documents if str(item.get("doc_id")) == str(doc_id)),
        None,
    )
    if document is None:
        raise ValueError(f"Document {doc_id!r} was not found in {input_path}.")

    sentences = document["sentences"]
    coref_sentences = (
        read_coref_sentences(
            coref_path, document["dataset"], doc_id, sentences
        )
        if coref_path is not None
        else [None] * len(sentences)
    )
    aligned_predictions = align_predictions_to_sentences(
        sentences, document.get("prediction", [])
    )
    aligned_gold_actions = align_gold_actions_to_sentences(
        sentences, document.get("gold_actions", [])
    )
    aligned_comparison_actions = read_comparison_actions(
        comparison_path, document["dataset"], doc_id, sentences
    )

    rows = [
        {
            "Sentence": sentence,
            "Corefed Sentence": coref_sentence,
            results_name: predictions,
            "Dataset Actions": gold_actions,
            "GPT3-to-Plan": comparison_actions,
        }
        for (
            sentence,
            coref_sentence,
            predictions,
            gold_actions,
            comparison_actions,
        ) in zip(
            sentences,
            coref_sentences,
            aligned_predictions,
            aligned_gold_actions,
            aligned_comparison_actions,
        )
    ]
    return pd.DataFrame(
        rows,
        columns=[
            "Sentence",
            "Corefed Sentence",
            results_name,
            "Dataset Actions",
            "GPT3-to-Plan",
        ],
    )


def dataframe_to_longtable(dataframe, caption, label):
    "Convert an aligned result DataFrame to the existing LaTeX longtable."
    results_name = get_results_column(dataframe)
    columns = ["Sentence", results_name, "Dataset Actions", "GPT3-to-Plan"]
    header_line = " & ".join(
        rf"\textbf{{{column}}}" for column in columns
    ) + r" \\"
    lines = [
        r"\begin{longtable}{@{}",
        r"    >{\raggedright\arraybackslash}p{0.34\textwidth}",
        r"    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}",
        r"    >{\raggedright\arraybackslash}p{0.308\textwidth}",
        r"    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}",
        r"    >{\raggedright\arraybackslash}p{0.165\textwidth}",
        r"    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}",
        r"    >{\raggedright\arraybackslash}p{0.165\textwidth}",
        r"@{}}",
        rf"\caption{{{escape_latex(caption)}}}",
        rf"\label{{{label}}} \\",
        "",
        r"% --- HEADER FOR FIRST PAGE ---",
        r"\toprule",
        header_line,
        r"\midrule",
        r"\endfirsthead",
        "",
        r"% --- HEADER FOR SUBSEQUENT PAGES ---",
        r"\multicolumn{4}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\",
        r"\toprule",
        header_line,
        r"\midrule",
        r"\endhead",
        "",
        r"% --- FOOTER FOR ALL PAGES EXCEPT THE LAST ---",
        r"\multicolumn{4}{r}{\textit{Continued on next page}} \\",
        r"\endfoot",
        "",
        r"% --- FOOTER FOR THE LAST PAGE ---",
        r"\bottomrule",
        r"\endlastfoot",
        "",
        r"% --- DATA ---",
        "",
    ]

    for position, (_, row) in enumerate(dataframe.iterrows()):
        sentence_text = escape_latex(row["Sentence"])
        coref_sentence = row["Corefed Sentence"]
        if coref_sentence is not None:
            sentence_text += (
                r"\par\noindent\rule{\linewidth}{0.4pt}\par "
                r"\textit{(corefed)} "
                + escape_latex(coref_sentence)
            )

        lines.append(
            sentence_text
            + " & "
            + format_action_list(row[results_name])
            + " & "
            + format_action_list(row["Dataset Actions"])
            + " & "
            + format_action_list(row["GPT3-to-Plan"])
            + r" \\"
        )
        if position < len(dataframe) - 1:
            lines.extend([r"\midrule", ""])

    lines.extend(["", r"\end{longtable}"])
    return "\n".join(lines) + "\n"


def json_result_to_longtable(
    input_path,
    results_name,
    doc_id,
    caption,
    label,
    comparison_path,
    coref_path=None,
):
    "Backward-compatible convenience wrapper around the DataFrame pipeline."
    dataframe = json_result_to_dataframe(
        input_path=input_path,
        results_name=results_name,
        doc_id=doc_id,
        comparison_path=comparison_path,
        coref_path=coref_path,
    )
    return dataframe_to_longtable(dataframe, caption=caption, label=label)

# NL2P Result

In [34]:
selected_dataset = "cooking"
doc_id = 62

dataframe1 = json_result_to_dataframe(
    input_path=PROJECT_ROOT / f"results/nl2p_1/{SELECTED_MODEL}/{selected_dataset}_nl2p_1_{SELECTED_MODEL}.json",
    results_name="NL2P",
    comparison_path=PROJECT_ROOT / f"results/gpt3_to_plan/{SELECTED_MODEL}/{selected_dataset}_gpt3_to_plan_{SELECTED_MODEL}.json",
    doc_id=doc_id,
)
display_wrapped_dataframe(dataframe1)

table1 = dataframe_to_longtable(
    dataframe1,
    caption=f"Extracted verb-argument predictions by {SELECTED_MODEL} prompted with NL2P",
    label=f"tab:{selected_dataset}_preds_nl2p",
)
print(table1)

,Sentence,Corefed Sentence,NL2P,Dataset Actions,GPT3-to-Plan
0,To begin with you need to put both the sugar and the evaporated milk into a saucepan and bring it to the boil slowly over a medium to low heat and always stir it in order to prevent the ingredients from sticking to the pan and burning,None,"[{'verb': 'put', 'arguments': ['the sugar', 'the evaporated milk', 'a saucepan']}, {'verb': 'bring to the boil', 'arguments': ['the sugar and the evaporated milk']}, {'verb': 'stir', 'arguments': ['the sugar and the evaporated milk']}, {'verb': 'stir', 'arguments': ['the sugar and the evaporated milk']}]","[{'verb': 'put', 'arguments': ['sugar', 'milk']}, {'verb': 'boil', 'arguments': ['it']}, {'verb': 'stir', 'arguments': ['it']}, {'verb': 'prevent', 'arguments': ['ingredients', 'sticking', 'burning']}]","[{'verb': 'put', 'arguments': ['sugar', 'milk']}, {'verb': 'bring', 'arguments': ['it', 'boil']}, {'verb': 'stir', 'arguments': ['it']}, {'verb': 'prevent', 'arguments': ['ingredients', 'sticking']}]"
1,When it comes to the boil you should continue to stir it for three minutes as you keep the heat going before moving to the next stage,None,[],"[{'verb': 'stir', 'arguments': ['it']}, {'verb': 'keep', 'arguments': ['heat']}, {'verb': 'moving', 'arguments': ['stage']}]","[{'verb': 'continue', 'arguments': ['stir']}, {'verb': 'keep', 'arguments': ['heat']}]"
2,After the three minutes are past you then need to take the mixture off the heat and add the chocolate chips and keep stirring so the mixture becomes nice and smooth,None,"[{'verb': 'take', 'arguments': ['the mixture']}, {'verb': 'add', 'arguments': ['the chocolate chips']}]","[{'verb': 'take', 'arguments': ['mixture']}, {'verb': 'add', 'arguments': ['chocolate', 'chips']}, {'verb': 'stirring', 'arguments': ['mixture']}]","[{'verb': 'take', 'arguments': ['mixture', 'heat']}, {'verb': 'add', 'arguments': ['chocolate', 'chips']}, {'verb': 'stir', 'arguments': ['mixture']}, {'verb': 'add', 'arguments': ['extract']}, {'verb': 'stir', 'arguments': ['everything']}]"
3,At this point you will then need to add in the extract that you plan on using and stir again so everything is very well mixed,None,"[{'verb': 'stir', 'arguments': ['the mixture']}, {'verb': 'add in', 'arguments': ['the extract that you plan on using']}, {'verb': 'stir', 'arguments': ['the mixture']}]","[{'verb': 'add', 'arguments': ['extract']}, {'verb': 'stir', 'arguments': ['everything']}, {'verb': 'mixed', 'arguments': ['everything']}]",[]
4,The final step is to put the mixture into the refrigerator for 90 minutes to two hours and then shape it into balls that are approximately one inch in diameter before rolling them in the nuts to ensure they are very well covered,None,"[{'verb': 'put', 'arguments': ['the mixture', 'the refrigerator']}, {'verb': 'shape', 'arguments': ['the mixture', 'balls that are approximately one inch in diameter']}, {'verb': 'roll', 'arguments': ['balls that are approximately one inch in diameter', 'the nuts']}]","[{'verb': 'put', 'arguments': ['mixture']}, {'verb': 'shape', 'arguments': ['mixture']}, {'verb': 'rolling', 'arguments': ['balls']}, {'verb': 'covered', 'arguments': ['balls']}]","[{'verb': 'put', 'arguments': ['mixture', 'refrigerator']}, {'verb': 'shape', 'arguments': ['it', 'balls']}, {'verb': 'rolling', 'arguments': ['balls', 'nuts']}]"
5,The truffles should then be placed back into the refrigerator until you are ready to serve them to ensure they keep their consistency as room temperature for a prolonged period of time will not be good for them,None,"[{'verb': 'placed back into', 'arguments': ['The truffles', 'the refrigerator']}, {'verb': 'serve', 'arguments': ['The truffles']}]","[{'verb': 'placed', 'arguments': ['truffles']}, {'verb': 'serve', 'arguments': ['truffles']}]","[{'verb': 'placed', 'arguments': ['truffles', 'refrigerator']}, {'verb': 'serve', 'arguments': ['truffles']}]"
6,That is basically how you make toasted almond truffles and you can see that there is nothin

\begin{longtable}{@{}
    >{\raggedright\arraybackslash}p{0.34\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.308\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.165\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.165\textwidth}
@{}}
\caption{Extracted verb-argument predictions by gpt-5.4 prompted with NL2P}
\label{tab:cooking_preds_nl2p} \\

% --- HEADER FOR FIRST PAGE ---
\toprule
\textbf{Sentence} & \textbf{NL2P} & \textbf{Dataset Actions} & \textbf{GPT3-to-Plan} \\
\midrule
\endfirsthead

% --- HEADER FOR SUBSEQUENT PAGES ---
\multicolumn{4}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\
\toprule
\textbf{Sentence} & \textbf{NL2P} & \textbf{Dataset Actions} & \textbf{GPT3-to-Plan} \\
\midrule
\endhead

% --- FOOTER FOR ALL PAGES EXCEPT THE LAST ---
\multicolumn{4}{r}{\text

# NL2P-Refined Result

In [35]:
selected_dataset = "cooking"
doc_id = 62

dataframe2 = json_result_to_dataframe(
    input_path=PROJECT_ROOT / f"results/nl2p_1_ablation/{SELECTED_MODEL}/{selected_dataset}_nl2p_1_ablation_{SELECTED_MODEL}.json",
    results_name="NL2P-Refined",
    comparison_path=PROJECT_ROOT / f"results/gpt3_to_plan/{SELECTED_MODEL}/{selected_dataset}_gpt3_to_plan_{SELECTED_MODEL}.json",
    doc_id=doc_id,
)
display_wrapped_dataframe(dataframe2)

table2 = dataframe_to_longtable(
    dataframe2,
    caption=f"Extracted verb-argument predictions by {SELECTED_MODEL} prompted with NL2P-Refined",
    label=f"tab:{selected_dataset}_preds_nl2p_refined",
)
print(table2)

,Sentence,Corefed Sentence,NL2P-Refined,Dataset Actions,GPT3-to-Plan
0,To begin with you need to put both the sugar and the evaporated milk into a saucepan and bring it to the boil slowly over a medium to low heat and always stir it in order to prevent the ingredients from sticking to the pan and burning,None,"[{'verb': 'put', 'arguments': ['both the sugar and the evaporated milk', 'a saucepan']}, {'verb': 'bring', 'arguments': ['the saucepan']}, {'verb': 'stir', 'arguments': ['the saucepan']}, {'verb': 'sticking', 'arguments': ['the ingredients', 'the pan']}, {'verb': 'burning', 'arguments': ['the ingredients']}]","[{'verb': 'put', 'arguments': ['sugar', 'milk']}, {'verb': 'boil', 'arguments': ['it']}, {'verb': 'stir', 'arguments': ['it']}, {'verb': 'prevent', 'arguments': ['ingredients', 'sticking', 'burning']}]","[{'verb': 'put', 'arguments': ['sugar', 'milk']}, {'verb': 'bring', 'arguments': ['it', 'boil']}, {'verb': 'stir', 'arguments': ['it']}, {'verb': 'prevent', 'arguments': ['ingredients', 'sticking']}]"
1,When it comes to the boil you should continue to stir it for three minutes as you keep the heat going before moving to the next stage,None,"[{'verb': 'comes to the boil', 'arguments': ['the saucepan']}, {'verb': 'continue to stir', 'arguments': ['the saucepan']}, {'verb': 'keep', 'arguments': ['the heat']}]","[{'verb': 'stir', 'arguments': ['it']}, {'verb': 'keep', 'arguments': ['heat']}, {'verb': 'moving', 'arguments': ['stage']}]","[{'verb': 'continue', 'arguments': ['stir']}, {'verb': 'keep', 'arguments': ['heat']}]"
2,After the three minutes are past you then need to take the mixture off the heat and add the chocolate chips and keep stirring so the mixture becomes nice and smooth,None,"[{'verb': 'take', 'arguments': ['the mixture']}, {'verb': 'add', 'arguments': ['the chocolate chips']}, {'verb': 'keep stirring', 'arguments': ['the mixture']}, {'verb': 'becomes', 'arguments': ['the mixture']}, {'verb': 'add in', 'arguments': ['the extract']}]","[{'verb': 'take', 'arguments': ['mixture']}, {'verb': 'add', 'arguments': ['chocolate', 'chips']}, {'verb': 'stirring', 'arguments': ['mixture']}]","[{'verb': 'take', 'arguments': ['mixture', 'heat']}, {'verb': 'add', 'arguments': ['chocolate', 'chips']}, {'verb': 'stir', 'arguments': ['mixture']}, {'verb': 'add', 'arguments': ['extract']}, {'verb': 'stir', 'arguments': ['everything']}]"
3,At this point you will then need to add in the extract that you plan on using and stir again so everything is very well mixed,None,"[{'verb': 'plan on using', 'arguments': ['the extract']}, {'verb': 'stir', 'arguments': ['everything']}]","[{'verb': 'add', 'arguments': ['extract']}, {'verb': 'stir', 'arguments': ['everything']}, {'verb': 'mixed', 'arguments': ['everything']}]",[]
4,The final step is to put the mixture into the refrigerator for 90 minutes to two hours and then shape it into balls that are approximately one inch in diameter before rolling them in the nuts to ensure they are very well covered,None,"[{'verb': 'put', 'arguments': ['the mixture', 'the refrigerator']}, {'verb': 'shape', 'arguments': ['the mixture', 'balls']}, {'verb': 'rolling', 'arguments': ['balls', 'the nuts']}, {'verb': 'covered', 'arguments': ['they']}]","[{'verb': 'put', 'arguments': ['mixture']}, {'verb': 'shape', 'arguments': ['mixture']}, {'verb': 'rolling', 'arguments': ['balls']}, {'verb': 'covered', 'arguments': ['balls']}]","[{'verb': 'put', 'arguments': ['mixture', 'refrigerator']}, {'verb': 'shape', 'arguments': ['it', 'balls']}, {'verb': 'rolling', 'arguments': ['balls', 'nuts']}]"
5,The truffles should then be placed back into the refrigerator until you are ready to serve them to ensure they keep their consistency as room temperature for a prolonged period of time will not be good for them,None,"[{'verb': 'be placed', 'arguments': ['The truffles', 'the refrigerator']}, {'verb': 'serve', 'arguments': ['them']}, {'verb': 'keep', 'arguments': ['they', 'their consistency']}]","[{'verb': 'placed', '

\begin{longtable}{@{}
    >{\raggedright\arraybackslash}p{0.34\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.308\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.165\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.165\textwidth}
@{}}
\caption{Extracted verb-argument predictions by gpt-5.4 prompted with NL2P-Refined}
\label{tab:cooking_preds_nl2p_refined} \\

% --- HEADER FOR FIRST PAGE ---
\toprule
\textbf{Sentence} & \textbf{NL2P-Refined} & \textbf{Dataset Actions} & \textbf{GPT3-to-Plan} \\
\midrule
\endfirsthead

% --- HEADER FOR SUBSEQUENT PAGES ---
\multicolumn{4}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\
\toprule
\textbf{Sentence} & \textbf{NL2P-Refined} & \textbf{Dataset Actions} & \textbf{GPT3-to-Plan} \\
\midrule
\endhead

% --- FOOTER FOR ALL PAGES EXCEPT THE L

# NL2P-Coref Result

In [36]:
selected_dataset = "cooking"
doc_id = 62

dataframe3 = json_result_to_dataframe(
    input_path=PROJECT_ROOT / f"results/nl2p_1_coref/{SELECTED_MODEL}/{selected_dataset}_nl2p_1_coref_{SELECTED_MODEL}.json",
    results_name="NL2P-Coref",
    comparison_path=PROJECT_ROOT / f"results/gpt3_to_plan/{SELECTED_MODEL}/{selected_dataset}_gpt3_to_plan_{SELECTED_MODEL}.json",
    coref_path=PROJECT_ROOT / f"data/coref_llm/{selected_dataset}_llm_coref.jsonl",
    doc_id=doc_id,
)
display_wrapped_dataframe(dataframe3)

table3 = dataframe_to_longtable(
    dataframe3,
    caption=f"Extracted verb-argument predictions by {SELECTED_MODEL} with Corefed Text",
    label=f"tab:{selected_dataset}_preds_nl2p_coref",
)
print(table3)

,Sentence,Corefed Sentence,NL2P-Coref,Dataset Actions,GPT3-to-Plan
0,To begin with you need to put both the sugar and the evaporated milk into a saucepan and bring it to the boil slowly over a medium to low heat and always stir it in order to prevent the ingredients from sticking to the pan and burning,To begin with you need to put both the sugar and the evaporated milk into a saucepan and bring the mixture to the boil slowly over a medium to low heat and always stir the mixture in order to prevent the ingredients from sticking to the pan and burning.,"[{'verb': 'put', 'arguments': ['the sugar', 'the evaporated milk', 'a saucepan']}, {'verb': 'bring', 'arguments': ['the mixture', 'the boil']}, {'verb': 'stir', 'arguments': ['the mixture']}]","[{'verb': 'put', 'arguments': ['sugar', 'milk']}, {'verb': 'boil', 'arguments': ['it']}, {'verb': 'stir', 'arguments': ['it']}, {'verb': 'prevent', 'arguments': ['ingredients', 'sticking', 'burning']}]","[{'verb': 'put', 'arguments': ['sugar', 'milk']}, {'verb': 'bring', 'arguments': ['it', 'boil']}, {'verb': 'stir', 'arguments': ['it']}, {'verb': 'prevent', 'arguments': ['ingredients', 'sticking']}]"
1,When it comes to the boil you should continue to stir it for three minutes as you keep the heat going before moving to the next stage,When the mixture comes to the boil you should continue to stir the mixture for three minutes as you keep the heat going before moving to the next stage.,"[{'verb': 'comes to the boil', 'arguments': ['the mixture']}, {'verb': 'continue to stir', 'arguments': ['the mixture']}, {'verb': 'keep', 'arguments': ['the heat']}]","[{'verb': 'stir', 'arguments': ['it']}, {'verb': 'keep', 'arguments': ['heat']}, {'verb': 'moving', 'arguments': ['stage']}]","[{'verb': 'continue', 'arguments': ['stir']}, {'verb': 'keep', 'arguments': ['heat']}]"
2,After the three minutes are past you then need to take the mixture off the heat and add the chocolate chips and keep stirring so the mixture becomes nice and smooth,After the three minutes are past you then need to take the mixture off the heat and add the chocolate chips and keep stirring so the mixture becomes nice and smooth.,"[{'verb': 'take', 'arguments': ['the mixture', 'the heat']}, {'verb': 'add', 'arguments': ['the chocolate chips']}, {'verb': 'keep stirring', 'arguments': ['the mixture']}, {'verb': 'add in', 'arguments': ['the extract']}]","[{'verb': 'take', 'arguments': ['mixture']}, {'verb': 'add', 'arguments': ['chocolate', 'chips']}, {'verb': 'stirring', 'arguments': ['mixture']}]","[{'verb': 'take', 'arguments': ['mixture', 'heat']}, {'verb': 'add', 'arguments': ['chocolate', 'chips']}, {'verb': 'stir', 'arguments': ['mixture']}, {'verb': 'add', 'arguments': ['extract']}, {'verb': 'stir', 'arguments': ['everything']}]"
3,At this point you will then need to add in the extract that you plan on using and stir again so everything is very well mixed,At this point you will then need to add in the extract that you plan on using and stir again so everything is very well mixed.,"[{'verb': 'stir', 'arguments': ['the mixture']}]","[{'verb': 'add', 'arguments': ['extract']}, {'verb': 'stir', 'arguments': ['everything']}, {'verb': 'mixed', 'arguments': ['everything']}]",[]
4,The final step is to put the mixture into the refrigerator for 90 minutes to two hours and then shape it into balls that are approximately one inch in diameter before rolling them in the nuts to ensure they are very well covered,The final step is to put the mixture into the refrigerator for 90 minutes to two hours and then shape the mixture into balls that are approximately one inch in diameter before rolling the balls in the nuts to ensure the balls are very well covered.,"[{'verb': 'put', 'arguments': ['the mixture', 'the refrigerator']}, {'verb': 'shape', 'arguments': ['the mixture', 'balls']}, {'verb': 'rolling', 'arguments': ['the balls', 'the nuts']}]","[{'verb': 'put', 'arguments': ['mixture']}, {'verb': 'shape', 'arguments': ['mixture']}, {'verb': 

\begin{longtable}{@{}
    >{\raggedright\arraybackslash}p{0.34\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.308\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.165\textwidth}
    @{\hspace{0.0035\textwidth}}|@{\hspace{0.0035\textwidth}}
    >{\raggedright\arraybackslash}p{0.165\textwidth}
@{}}
\caption{Extracted verb-argument predictions by gpt-5.4 with Corefed Text}
\label{tab:cooking_preds_nl2p_coref} \\

% --- HEADER FOR FIRST PAGE ---
\toprule
\textbf{Sentence} & \textbf{NL2P-Coref} & \textbf{Dataset Actions} & \textbf{GPT3-to-Plan} \\
\midrule
\endfirsthead

% --- HEADER FOR SUBSEQUENT PAGES ---
\multicolumn{4}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\
\toprule
\textbf{Sentence} & \textbf{NL2P-Coref} & \textbf{Dataset Actions} & \textbf{GPT3-to-Plan} \\
\midrule
\endhead

% --- FOOTER FOR ALL PAGES EXCEPT THE LAST ---
\multic